# Tu copia personal · Aula UTP

Este cuaderno se abre desde GitHub. En Colab elige **Archivo → Guardar una copia en Drive** antes de conservar cambios. Renombra esa copia y ejecuta de arriba abajo. Cada estudiante trabaja en su propio archivo y entorno.

Los ejemplos son didácticos: sigue la sección indicada por Cristian para esta clase. Las comprobaciones son voluntarias, sin entregas ni notas.

# Lectura guiada · Machine Learning UTP
Cristian Alvarado · Ejecuta de arriba hacia abajo. No hay tareas ni entregas. Los ejemplos se revisan con el docente.

## Ruta de esta práctica · ML · Encuentro 3 · Ampliación de agrupación

**En el aula empezamos por DBSCAN:** su práctica principal está en **Laboratorio ML explicado, bloque 4**. Este cuaderno amplía la misma pregunta —reunir casos parecidos— con dos métodos distintos: **K-Means** y **Mean-Shift**. No hace falta completarlos para entender el ejemplo principal ni se espera que produzcan sus mismos grupos.

**El puente entre ejemplos:** en el aula comparas seis centros de acopio, el explorador tiene 29 perfiles y DBSCAN usa 123 puntos inventados. Aquí cada bloque genera 450 fincas ficticias con seis medidas. Cambian datos y método; se conserva la pregunta por el parecido. No mezcles sus conteos o números de grupo.

**Antes de comenzar:** en Colab elige Archivo → Guardar una copia en Drive. Ejecuta un bloque completo de arriba abajo; el segundo genera sus propios datos y se puede estudiar aparte. Una celda de código es una instrucción que se ejecuta con ▶. La salida aparece debajo; leer una figura no vuelve a ajustar el modelo.

**Entorno:** CPU es suficiente. No actives recursos de pago ni claves de API. Si Colab interrumpe la sesión, los resultados guardados en pantalla no recuperan las variables: vuelve a ejecutar el bloque completo en tu copia.

### Nombres que usaremos al agrupar

Estas palabras ya aparecen en esta práctica. El nombre completo ayuda a leer el código; no añade una tarea nueva.

- **DBSCAN** — DBSCAN: Density-Based Spatial Clustering of Applications with Noise (Agrupación espacial basada en densidad con casos no agrupados). Método que forma grupos conectando zonas con suficientes vecinos. eps fija la distancia máxima entre vecinos y min_samples el mínimo para que un punto sea núcleo, contando al propio punto. No se elige de antemano el número de grupos; −1 identifica puntos sin grupo con esa configuración. El radio no limita la distancia entre todos los miembros de un grupo y quedar aparte no demuestra mala calidad. [Fuente](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.DBSCAN.html)

- **K-Means** — K-Means: K medias. Método que recibe un número k de grupos y alterna entre asignar cada caso al centro más cercano y recalcular cada centro como la media de los casos asignados. k es una cantidad elegida, no una sigla. Busca reducir distancias cuadradas a los centros; escala e inicialización importan y un grupo no equivale automáticamente a una categoría real. [Fuente](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html)

- **Mean-Shift** — Mean-Shift: Desplazamiento hacia medias locales. Método que desplaza centros candidatos hacia la media de los puntos de su vecindario y reúne los que convergen a zonas de concentración próximas. No pide k de entrada, pero sí una escala de vecindad llamada bandwidth, elegida o estimada. Los grupos dependen de esa escala y de la preparación, no son clases verdaderas demostradas. [Fuente](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.MeanShift.html)

- **Estandarización** — Estandarización: Centrar y expresar en unidades de desviación estándar. Resta a cada valor la media de entrenamiento y divide entre su desviación estándar de entrenamiento: z=(valor−media)/desviación. Escalado es el nombre más general de cambiar la escala; centrar solo resta la media. La desviación estándar mide dispersión y es la raíz de la varianza; tras dividir se pierden las unidades físicas originales. Si una columna es constante, StandardScaler evita dividir entre cero; al predecir se reutilizan las estadísticas de entrenamiento. [Fuente](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html)


## 1. K-Means: organizar fincas por parecido

**Situación:** una cooperativa quiere empezar una revisión por fincas con condiciones semejantes. No está decidiendo cuál es mejor ni recomendando fertilizar. Un caso es una finca ficticia; cada medida describe algo de ella.

Se inventan 450 fincas: 150, 160 y 140 alrededor de tres perfiles que elegimos al construir los datos. Seis medidas: lluvia anual en mm, temperatura media en °C, pH (escala de acidez, sin unidad), materia orgánica en %, altitud en metros sobre el nivel del mar y pendiente en %. Son números para estudiar el método, no un muestreo agrícola validado.

**Antes de la fórmula:** un grupo reúne fincas parecidas; su centroide es el promedio de las medidas de sus integrantes. **k** es cuántos grupos pedimos. Con k=3 pedimos tres centros; el método alterna asignar fincas al centro más cercano y recalcular esos promedios.

El bloque hace esa operación con seis medidas a la vez. Por eso el primer gráfico de puntos usa un resumen en dos ejes para poder dibujarlo; el resumen no cambia las seis medidas con las que se hicieron los grupos.

### Qué significa escalar y cómo leer las primeras curvas

Lluvia puede valer 1800 y pH, 6. Comparar directamente esos números daría mucho peso a la lluvia solo por su unidad. **Estandarizar** expresa cada medida como distancia respecto a su promedio, medida en desviaciones. La desviación resume cuánto suelen separarse los valores de su media. Ejemplo inventado: si una medida vale 700, su media es 600 y su desviación 100, su valor estandarizado es (700−600)/100 = 1: está una desviación por encima. La celda aprende medias y escalas con estas 450 fincas.

**Lee una salida a la vez:**

1. En la tabla inicial, cada fila es una finca y el encabezado de cada columna nombra una medida. `df.head()` enseña las primeras cinco; no reduce el conjunto a cinco.
2. En la curva del codo, el eje horizontal es k. La **inercia** suma las separaciones al cuadrado de cada finca a su centro. Menor inercia significa grupos más compactos según esta escala; pedir más grupos suele reducirla incluso si dejan de ser útiles. El «codo» es un cambio de ritmo que se busca visualmente, no una respuesta automática.
3. **Silhouette** compara cuánto se parece una finca a su propio grupo frente al grupo alternativo más cercano. Cerca de 1 indica separación mayor, cerca de 0 un límite poco claro y un valor negativo una asignación dudosa según las distancias. No es porcentaje de aciertos.

El detalle de esa cuenta usa a = distancia media al propio grupo y b = distancia media al otro grupo más cercano: (b−a)/máximo(a,b). Después se promedian las fincas.

**Tip:** aquí fijamos k=3 porque construimos tres perfiles; las curvas permiten investigarlo, no prueban tres clases naturales. No cambies simultáneamente k, las medidas y su escala: no sabrías qué produjo la diferencia.

In [ ]:
from IPython.display import display
# ================================================================
# CLUSTERING CON K-MEANS (Colab listo para pegar)
# Caso didáctico: segmentación de fincas agrícolas por características
# ================================================================
# ¿Qué hace este notebook?
# 1) Genera un dataset sintético "realista" de fincas con variables agro.
# 2) Normaliza (es crucial para K-means).
# 3) Selecciona k con método del codo (inertia) y silhouette.
# 4) Entrena K-means y proyecta a 2D con PCA para visualizar.
# 5) Interpreta clústeres: centroides en unidades originales y “perfiles”.
# 6) Muestra cómo asignar un nuevo ejemplo al grupo más cercano.
#
# ------------------------------------------------
# ¿Cómo funciona K-means? (resumen algorítmico)
# ------------------------------------------------
# - Entrada: datos numéricos y un número de clústeres k.
# - Inicialización: elige k centroides (p. ej. k-means++).
# - Paso de asignación: asigna cada punto al centroide más cercano (distancia euclídea).
# - Paso de actualización: recalcula cada centroide como la media de los puntos de su clúster.
# - Repite asignar/actualizar hasta converger (cambios mínimos o máx. iteraciones).
# - Objetivo: minimizar la suma de distancias cuadráticas dentro de cada clúster (inertia).
#
# Interpretación práctica:
# - Cada clúster agrupa observaciones similares; el centroide resume el “perfil medio”.
# - Elegir k: buscamos equilibrio entre compacidad (inertia baja) y separación (silhouette alta).
# ================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# ----------------------------
# 1) Generación del dataset
# ----------------------------
# Variables (todas continuas):
# - lluvia_mm                (anual)
# - temp_media_c             (°C)
# - ph_suelo                 (0-14)
# - materia_organica_pct     (%)
# - altitud_m                (msnm)
# - pendiente_pct            (%)
#
# Creamos 3 “tipos” de fincas con medias diferentes para que K-means los detecte.

rng = np.random.default_rng(7)

n1, n2, n3 = 150, 160, 140  # tamaños por grupo
# Grupo 1: tierras bajas, clima cálido, más materia orgánica
g1 = np.column_stack([
    rng.normal(1800, 180, n1),  # lluvia_mm
    rng.normal(25.0, 1.2, n1),  # temp_media_c
    rng.normal(6.3, 0.3, n1),   # ph_suelo
    rng.normal(5.5, 0.8, n1),   # materia_organica_pct
    rng.normal(600, 120, n1),   # altitud_m
    rng.normal(4, 1.5, n1)      # pendiente_pct
])

# Grupo 2: altiplano templado, pH algo mayor, pendiente media
g2 = np.column_stack([
    rng.normal(1400, 160, n2),
    rng.normal(19.0, 1.0, n2),
    rng.normal(6.8, 0.25, n2),
    rng.normal(3.8, 0.6, n2),
    rng.normal(1500, 200, n2),
    rng.normal(8, 2.0, n2)
])

# Grupo 3: alta montaña fresca, suelos más ácidos y pendientes altas
g3 = np.column_stack([
    rng.normal(1100, 150, n3),
    rng.normal(15.0, 1.1, n3),
    rng.normal(5.8, 0.25, n3),
    rng.normal(3.2, 0.5, n3),
    rng.normal(2400, 220, n3),
    rng.normal(15, 3.0, n3)
])

X = np.vstack([g1, g2, g3])
df = pd.DataFrame(X, columns=[
    "lluvia_mm", "temp_media_c", "ph_suelo",
    "materia_organica_pct", "altitud_m", "pendiente_pct"
])

print("Dimensiones del dataset:", df.shape)
display(df.head())

# ----------------------------
# 2) Normalización
# ----------------------------
# K-means usa distancia euclídea; si no escalamos, una variable con unidades grandes (p. ej. altitud)
# dominará el agrupamiento. Usamos StandardScaler (media 0, var 1).
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)

# ----------------------------
# 3) Selección de k
# ----------------------------
# a) Método del codo: graficamos inertia vs. k y buscamos el “codo”.
# b) Silhouette: valores más altos (~0.5-1) indican mejor separación entre clústeres.

k_values = range(2, 9)
inertias = []
silhouettes = []

for k in k_values:
    km = KMeans(n_clusters=k, n_init=10, random_state=7)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

plt.figure(figsize=(6,4))
plt.plot(k_values, inertias, marker='o')
plt.title("Método del codo (Inertia vs k)")
plt.xlabel("k")
plt.ylabel("Inertia (suma de distancias al cuadrado)")
plt.grid(True)
plt.show()

plt.figure(figsize=(6,4))
plt.plot(k_values, silhouettes, marker='o')
plt.title("Índice de Silhouette vs k")
plt.xlabel("k")
plt.ylabel("Silhouette")
plt.grid(True)
plt.show()

print("Tabla k / inertia / silhouette")
print(pd.DataFrame({"k": k_values, "inertia": inertias, "silhouette": silhouettes}))

# Tip: para este dataset, esperamos k≈3 (porque lo generamos así),
# pero en datos reales NO sabemos k; por eso miramos ambas curvas y contexto de negocio.

# ----------------------------
# 4) Ajuste con k elegido para este ejemplo (aquí tomamos k=3)
# ----------------------------
k_opt = 3
kmeans = KMeans(n_clusters=k_opt, n_init=10, random_state=7)
labels = kmeans.fit_predict(X_scaled)

df_clusters = df.copy()
df_clusters["cluster"] = labels

print("\nConteo por clúster")
print(df_clusters["cluster"].value_counts().sort_index())

# ----------------------------
# 5) Proyección 2D con PCA para visualizar clústeres
# ----------------------------
# PCA no “mejora” el clustering; solo reduce a 2D para ver separación aproximada.
pca = PCA(n_components=2, random_state=7)
X_2d = pca.fit_transform(X_scaled)
centroids_2d = pca.transform(kmeans.cluster_centers_)

plt.figure(figsize=(6,5))
for c in range(k_opt):
    plt.scatter(X_2d[labels==c, 0], X_2d[labels==c, 1], s=18, alpha=0.7, label=f"Cluster {c}")
plt.scatter(centroids_2d[:,0], centroids_2d[:,1], s=180, marker='X', edgecolor='k', label="Centroides")
plt.title("Clústeres en 2D (PCA)")
plt.xlabel("Componente principal 1")
plt.ylabel("Componente principal 2")
plt.legend()
plt.grid(True)
plt.show()

# ----------------------------
# 6) Interpretación de clústeres
# ----------------------------
# a) Centroides en espacio escalado -> difícil de leer.
# b) Convertimos centroides a unidades originales con inverse_transform.
centroids_original = pd.DataFrame(
    scaler.inverse_transform(kmeans.cluster_centers_),
    columns=df.columns
).round(2)
centroids_original.index = [f"Cluster {i}" for i in range(k_opt)]

print("\nCentroides en unidades originales (interpretables):")
display(centroids_original)

# Perfil de cada clúster: medias por variable (debería ser similar a los centroides)
perfil = df_clusters.groupby("cluster").mean(numeric_only=True).round(2)
perfil.index = [f"Cluster {i}" for i in perfil.index]
print("\nPerfil (media por variable y clúster):")
display(perfil)

# Para resaltar variables más “diferenciadoras” por clúster,
# calculamos z-scores de los centroides vs. media global.
mu = df.mean()
sigma = df.std(ddof=0)

z_centroids = ((centroids_original - mu) / sigma).round(2)
print("\nZ-scores de centroides (positivo = por encima del promedio global):")
display(z_centroids)

# Guía rápida de lectura:
print("""
================= CÓMO LEER LOS RESULTADOS =================
1) Gráfica PCA:
   - Nubes separadas = clústeres bien diferenciados.
   - El marcador X indica el centroide (promedio) de cada grupo.

2) Centroides en unidades originales:
   - Interpretación directa. Ejemplo: si 'Cluster 2' tiene 'altitud_m' = 2300,
     ese grupo agrupa fincas de alta montaña.

3) Z-scores de centroides:
   - Valores > 0: variable por encima del promedio global del dataset en ese clúster.
   - Valores < 0: por debajo del promedio.
   - Útiles para describir “rasgos distintivos” de cada segmento.

4) Reglas prácticas:
   - Si un clúster muestra lluvia alta + temp alta + baja altitud,
     podrías asociarlo a tierras bajas cálidas.
   - Si otro clúster tiene pH más ácido + alta altitud + alta pendiente,
     podrías asociarlo a alta montaña con suelos más ácidos, etc.

5) ¿k correcto?
   - Revisa el “codo” (inertia) y el valor de silhouette.
   - Confirma con criterio de negocio: ¿tiene sentido operativo diferenciar esos grupos?
=============================================================
""")

# ----------------------------
# 7) Asignar un nuevo ejemplo a un grupo
# ----------------------------
# Proceso: (i) escalar con el MISMO scaler, (ii) usar kmeans.predict, (iii) interpretar.
nuevo = pd.DataFrame([{
    "lluvia_mm": 1350,
    "temp_media_c": 18.5,
    "ph_suelo": 6.9,
    "materia_organica_pct": 3.7,
    "altitud_m": 1600,
    "pendiente_pct": 7.0
}])

nuevo_scaled = scaler.transform(nuevo)
cluster_pred = kmeans.predict(nuevo_scaled)[0]

print("Nuevo ejemplo:")
display(nuevo)
print(f"Asignado a: Cluster {cluster_pred}")

print("""
Interpretación:
- Compara las variables del nuevo ejemplo con los centroides en unidades originales.
- Describe el perfil del grupo asignado en esta simulación; su número es un identificador.
- Una decisión de manejo requeriría datos representativos y evaluación agronómica adicional.
""")


### Después de K-Means: vuelve del dibujo a una finca

En la figura PCA, cada punto sigue siendo una finca y la cruz su centro de grupo proyectado. **PC1 y PC2** son combinaciones de las seis medidas para dibujar; no significan «lluvia» y «temperatura» ni conservan necesariamente todas las distancias. No deduzcas una decisión de manejo mirando únicamente una nube de color.

Busca el grupo asignado a `nuevo` en la última salida. En la tabla de centros en unidades originales, localiza esa misma etiqueta y describe dos medidas: «su altitud media es… metros y su lluvia… mm». `inverse_transform` deshace la escala para facilitar esa lectura; no descubre causas ni añade medidas.

**Tres operaciones de código:** `fit` ajusta centros con ejemplos; `transform` aplica una transformación ya aprendida; `predict` asigna el nuevo caso sin volver a aprender los centros. El nuevo caso usa **el mismo scaler**; aprender otro con una sola fila borraría la comparación.

**Prueba pequeña:** guarda primero la etiqueta y el conteo de cada grupo. En tu copia cambia solo `k_opt` de 3 a 4 y vuelve a ejecutar todo este bloque desde su semilla. Debe haber cuatro centros; cambian las asignaciones, no las fincas simuladas. Compara los perfiles, no el nombre numérico del grupo.

## 2. Mean-Shift: buscar concentraciones sin pedir k

**Situación:** ahora queremos que los centros aparezcan donde se acumulan casos parecidos, sin solicitar tres de antemano. El bloque genera **otras 450 fincas ficticias**; no son exactamente las mismas del bloque 1. Comparar sus resultados no es una prueba justa de cuál algoritmo es mejor.

**Densidad** significa aquí cuántos casos se encuentran cerca. Mean-Shift desplaza candidatos a centro hacia el promedio de los vecinos. **Bandwidth** es la cercanía que usa en cada paso. Cambiarla puede unir o separar concentraciones; no pedir k no elimina decisiones de configuración.

Ejecuta el bloque entero y busca cuántos grupos obtuvo. Sus números siguen siendo identificadores, no niveles de calidad.

### Qué calcula el radio y qué no demuestra

Se estandarizan las medidas igual que en el primer bloque: la cercanía se mide en ese espacio preparado, no directamente en mm o metros. El método `estimate_bandwidth` propone un radio usando distancias de una muestra; `quantile=0.15` es una elección del ejercicio, no «15 % de aciertos». `n_samples=300` toma una submuestra para acelerar esa estimación.

**Cómo leer los resultados:** una cruz es un centro encontrado. Los colores identifican grupos. El promedio de todas las fincas finalmente asignadas a un centro puede diferir de ese centro; por eso las dos tablas no tienen que ser idénticas. La tabla devuelta a unidades originales permite describir el perfil sin interpretar directamente coordenadas abstractas.

**Tip:** si aparece un único grupo, no es por sí solo un fallo. Conserva los datos y compara el efecto de otro radio; después pregunta qué revisión podría organizarse con ese resultado. No fuerces un número de grupos solo para que el dibujo resulte atractivo.

In [ ]:
from IPython.display import display
# =====================================================================
# CLUSTERING CON MEAN-SHIFT (Colab listo para pegar)
# Caso didáctico: segmentación de fincas agrícolas por características
# =====================================================================
# ¿Qué hace este notebook?
# 1) Genera un dataset sintético "realista" de fincas con variables agro.
# 2) Normaliza (crítico para distancias).
# 3) Estima automáticamente el parámetro clave: bandwidth (radio del kernel).
# 4) Ajusta Mean-Shift (encuentra k dinámicamente, sin fijar k).
# 5) Visualiza en 2D (PCA), muestra centros y tamaños de clúster.
# 6) Interpreta: centroides en unidades originales y z-scores.
# 7) “Predice” nuevos puntos por asignación al centro más cercano (ver nota).
#
# ---------------------------------------------------------------------
# ¿Cómo funciona Mean-Shift? (resumen algorítmico)
# ---------------------------------------------------------------------
# - La implementación usada de scikit-learn utiliza un kernel plano: calcula
#   la media de los puntos dentro de una vecindad de radio bandwidth.
# - "Desplaza" cada punto gradualmente hacia regiones de mayor densidad
#   (modos de la distribución). Puntos que convergen al mismo modo forman un clúster.
# - No requiere k. La cantidad de clústeres emerge de la densidad y del bandwidth.
#
# Trade-offs:
# - Bandwidth pequeño => muchos clústeres (sobre-segmentación).
# - Bandwidth grande  => pocos clústeres (sub-segmentación).
# - Complejidad alta en datasets grandes; conviene escalar y muestrear para estimar bandwidth.
# =====================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import MeanShift, estimate_bandwidth
from sklearn.metrics import silhouette_score

# ----------------------------
# 1) Generación del dataset
# ----------------------------
# Variables continuas:
# - lluvia_mm, temp_media_c, ph_suelo, materia_organica_pct, altitud_m, pendiente_pct
# Creamos 3 “regímenes” para que Mean-Shift pueda detectarlos sin fijar k.

rng = np.random.default_rng(11)
n1, n2, n3 = 150, 160, 140

g1 = np.column_stack([
    rng.normal(1750, 170, n1),  # lluvia_mm
    rng.normal(24.5, 1.2, n1),  # temp_media_c
    rng.normal(6.4, 0.25, n1),  # ph_suelo
    rng.normal(5.2, 0.7, n1),   # materia_organica_pct
    rng.normal(700, 110, n1),   # altitud_m
    rng.normal(5, 1.7, n1)      # pendiente_pct
])

g2 = np.column_stack([
    rng.normal(1450, 160, n2),
    rng.normal(19.5, 1.0, n2),
    rng.normal(6.9, 0.22, n2),
    rng.normal(3.9, 0.6, n2),
    rng.normal(1550, 190, n2),
    rng.normal(8.5, 2.0, n2)
])

g3 = np.column_stack([
    rng.normal(1150, 150, n3),
    rng.normal(15.2, 1.0, n3),
    rng.normal(5.7, 0.25, n3),
    rng.normal(3.1, 0.5, n3),
    rng.normal(2450, 220, n3),
    rng.normal(15.5, 3.0, n3)
])

X = np.vstack([g1, g2, g3])
df = pd.DataFrame(X, columns=[
    "lluvia_mm", "temp_media_c", "ph_suelo",
    "materia_organica_pct", "altitud_m", "pendiente_pct"
])

print("Dimensiones del dataset:", df.shape)
display(df.head())

# ----------------------------
# 2) Normalización
# ----------------------------
# Distancias euclídeas sensibles a escala -> StandardScaler (media 0, var 1).
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)

# ----------------------------
# 3) Estimación de bandwidth
# ----------------------------
# estimate_bandwidth usa percentiles de distancias para sugerir el radio del kernel.
# - quantile ∈ (0,1): percentil de distancias; más bajo => ventanas pequeñas => más clusters.
# - sample_size: submuestra para acelerar el cálculo en datasets grandes.
# Ajusta quantile si ves “muchos” o “pocos” clústeres.

bandwidth = estimate_bandwidth(
    X_scaled,
    quantile=0.15,      # valor típico entre 0.1 y 0.3
    n_samples=300,      # submuestra para velocidad
    random_state=11
)

print(f"Bandwidth estimado: {bandwidth:.4f}")

# Si bandwidth es muy pequeño o muy grande, ajusta el quantile y re-ejecuta.

# ----------------------------
# 4) Ajuste de Mean-Shift
# ----------------------------
# bin_seeding=True: inicializa con una cuadrícula de bins para acelerar.
# cluster_all=True: etiqueta todos los puntos; si False, puntos alejados reciben -1.
ms = MeanShift(
    bandwidth=bandwidth,
    bin_seeding=True,
    cluster_all=True
)
labels = ms.fit_predict(X_scaled)  # ajusta el modelo y devuelve sus etiquetas de entrenamiento
centers_scaled = ms.cluster_centers_
n_clusters_ = len(np.unique(labels))

print(f"Número de clústeres detectados: {n_clusters_}")
df_ms = df.copy()
df_ms["cluster"] = labels
print("\nConteo por clúster:")
print(df_ms["cluster"].value_counts().sort_index())

# Opcional: calidad de la partición (si hay >=2 clústeres)
if 2 <= n_clusters_ < len(X_scaled):
    sil = silhouette_score(X_scaled, labels)
    print(f"\nSilhouette (>=2 clústeres): {sil:.3f}")
else:
    print("\nSilhouette no aplicable (requiere entre 2 y n−1 grupos). Ajusta 'quantile' para más granularidad.")

# ----------------------------
# 5) Visualización 2D (PCA)
# ----------------------------
pca = PCA(n_components=2, random_state=11)
X_2d = pca.fit_transform(X_scaled)
centers_2d = pca.transform(centers_scaled)

plt.figure(figsize=(6,5))
for c in range(n_clusters_):
    plt.scatter(X_2d[labels==c, 0], X_2d[labels==c, 1], s=18, alpha=0.7, label=f"Cluster {c}")
plt.scatter(centers_2d[:,0], centers_2d[:,1], s=180, marker='X', edgecolor='k', label="Centros")
plt.title("Mean-Shift en 2D (PCA)")
plt.xlabel("Componente principal 1")
plt.ylabel("Componente principal 2")
plt.legend()
plt.grid(True)
plt.show()

# ----------------------------
# 6) Interpretación de clústeres
# ----------------------------
# Convertimos centros a unidades originales para lectura directa.
centers_original = pd.DataFrame(
    scaler.inverse_transform(centers_scaled),
    columns=df.columns
).round(2)
centers_original.index = [f"Cluster {i}" for i in range(n_clusters_)]

print("\nCentros (unidades originales):")
display(centers_original)

perfil = df_ms.groupby("cluster").mean(numeric_only=True).round(2)
perfil.index = [f"Cluster {i}" for i in perfil.index]
print("\nPerfil de variables por clúster (media):")
display(perfil)

mu = df.mean()
sigma = df.std(ddof=0)
z_centers = ((centers_original - mu) / sigma).round(2)
print("\nZ-scores de centros (positivo = sobre promedio global):")
display(z_centers)

print("""
================= CÓMO LEER LOS RESULTADOS =================
1) Mean-Shift detecta clústeres a partir de la densidad. No fija k.
   El número de clústeres depende del 'bandwidth' (ventana del kernel).

2) Gráfica PCA:
   - Nubes separadas y centros (X) indican segmentos bien diferenciados.

3) Centros en unidades originales:
   - Permiten describir cada segmento con variables del dominio (lluvia, altitud, etc.).

4) Z-scores:
   - Identifican variables “salientes” por clúster (>> 0 o << 0).

5) Si devuelve 1 solo clúster:
   - Reduce 'quantile' (ej. 0.10) para un bandwidth más pequeño y mayor granularidad.

6) Costo computacional:
   - Mean-Shift escala mal con n grande. Para datos masivos, considerar mini-batch K-means,
     HDBSCAN o un pre-agrupamiento + Mean-Shift en centros.
=============================================================
""")

# ----------------------------
# 7) “Predicción” para nuevos puntos
# ----------------------------
# MeanShift.predict asigna cada punto al centro aprendido más cercano.
# Aplicamos el MISMO scaler del ajuste; no lo volvemos a aprender.
# Reentrenar con más observaciones sería un nuevo ajuste, con otros centros posibles.

nuevo = pd.DataFrame([{
    "lluvia_mm": 1380,
    "temp_media_c": 18.8,
    "ph_suelo": 6.85,
    "materia_organica_pct": 3.6,
    "altitud_m": 1620,
    "pendiente_pct": 7.5
}])

asignacion = ms.predict(scaler.transform(nuevo))[0]
print("Nuevo ejemplo:")
display(nuevo)
print(f"Asignado al centro aprendido más cercano: Cluster {asignacion}")

print("""
Interpretación:
- Compara el nuevo registro con el centro correspondiente y con el perfil del grupo.
  Un centro de Mean-Shift no tiene por qué coincidir con la media de todo el grupo asignado.
- Esta asignación describe la simulación con centros fijos; no valida decisiones agronómicas.
  Reentrenar al añadir datos puede cambiar centros y grupos: sería otro modelo.
""")

# ----------------------------
# 8) Hiperparámetros clave para ajustar
# ----------------------------
print("""
Sugerencias de tuning:
- estimate_bandwidth.quantile: 0.10–0.30 suele ser buen rango. Más bajo => más clústeres.
- MeanShift(bin_seeding=True): acelera en alta dimensión.
- Aquí estandarizamos porque las unidades difieren; la elección de escala define qué similitud medimos.
- Si el dataset es muy grande: muestrear para estimar bandwidth o considerar alternativas.
""")


### Después de Mean-Shift: describe antes de actuar

La salida «Asignado al centro aprendido más cercano» ubica la finca nueva entre los centros ya calculados. No demuestra que esa finca comparta una categoría biológica real. Comprueba su perfil con las seis medidas originales y conserva una frase sobre lo que todavía tendrías que revisar.

**Tip de comparación:** cambia únicamente `quantile` en tu copia, por ejemplo de 0,15 a 0,20, y ejecuta el segundo bloque entero desde su propia semilla. Anota el bandwidth obtenido y el número de grupos. Puede cambiar el número; no tiene que coincidir con k=3 de K-Means. El conteo total debe seguir sumando 450 porque `cluster_all=True` asigna todas estas filas.